# 📊 Módulo de Extração e Engenharia de Dados Relacionais

Este caderno converte o backup relacional bruto do ERP (`.sql`) no arquivo tabular que alimenta o pipeline de modelagem em `Chopp_Risco_EDA_ML_v1.3.ipynb`.

---

## 🎯 Princípio: extrair o necessário, não o disponível

O dump contém **133 tabelas** e centenas de colunas cada — `TB_PEDIDO_ITEM` sozinha tem 200, quase todas fiscais (IBS, CBS, ICMS-ST, PIS, COFINS). Nenhuma delas entra num modelo de risco de crédito.

A versão anterior deste notebook exportava tudo, produzindo um consolidado de **300 colunas e ~17 MB** do qual o pipeline de ML usava menos de 10%. Agora um **contrato de colunas** (célula seguinte) declara explicitamente cada campo consumido, com a justificativa de uso ao lado, e a projeção é aplicada **durante o parse** — as colunas fora do contrato nunca chegam à memória.

## 🏗️ Arquitetura do Processo

1. **Contrato de colunas:** dicionário `COLUNAS_NECESSARIAS`, rastreado campo a campo até o ponto de uso no notebook de ML. É a interface entre os dois cadernos: feature nova lá ⇒ coluna de origem aqui.

2. **Parser com projeção:** varredura por `RegEx` sobre os `INSERT`, neutralizando os `CAST(...)` do SQL Server e alinhando defensivamente linhas encurtadas por `NULL`. Só as posições do contrato são materializadas.

3. **Montagem das views:** reproduz em pandas o que as `vw_calculo_score_*` fazem no SQL Server, unindo item↔pedido, parcela↔título e comodato↔bem, e exporta as três views mais as dimensões.

4. **Auditoria de integridade:** falha explicitamente se uma view sair vazia ou se a ponte `ID_CLIENTE ↔ ID_PESSOA` tiver cobertura baixa — condições que não quebram o notebook de ML, mas corrompem a variável-alvo em silêncio.

# 📊 Pipeline de Extração e Unificação de Dados do ERP

Este notebook tem como objetivo processar um backup relacional bruto (`.sql`) do sistema de gestão, extraindo as tabelas dimensionais e transacionais essenciais, unificando-as por meio de cruzamentos relacionais (*Left Joins*) e gerando um dataset consolidado otimizado para Machine Learning.

In [4]:
# ==============================================================================
# 1. CONFIGURAÇÃO E CONTRATO DE COLUNAS
# ==============================================================================
import re
import csv
import os
import pandas as pd
from pathlib import Path

# Arquivo SQL de entrada
SQL_FILE = Path(r"C:\_Pessoas\Bruno.Araujo\CDN_ProjetoIntegrador6\Dados\DB_POWER_SYS202608261200_260828.sql")

# Diretório de saída
OUTPUT_DIR = Path(r"C:\_Pessoas\Bruno.Araujo\CDN_ProjetoIntegrador6\Desenvolvimento\data") 

LEITURA_PARAMS = {
    "sep": ";",
    "encoding": "utf-8-sig",
    "low_memory": False,
}

ESCRITA_PARAMS = {
    "sep": ";",
    "encoding": "utf-8-sig",
    "index": False,
}

# ══════════════════════════════════════════════════════════════════════════════
#  CONTRATO DE COLUNAS — o que o notebook de ML realmente consome
#
#  O dump traz 133 tabelas e centenas de colunas por tabela: TB_PEDIDO_ITEM
#  sozinha tem 200, quase todas fiscais (IBS/CBS/ICMS-ST/PIS/COFINS). Nenhuma
#  entra no modelo de risco. Exportar tudo produzia um consolidado de 300
#  colunas e ~17 MB, do qual o pipeline de ML usava menos de 10%.
#
#  Cada coluna abaixo foi rastreada até o ponto de uso em
#  Chopp_Risco_EDA_ML_v1.3.ipynb. Se uma coluna não está aqui, é porque nenhuma
#  célula daquele notebook a lê. Ao acrescentar uma feature nova lá, acrescente
#  a coluna de origem aqui — este dicionário é o contrato entre os dois cadernos.
# ══════════════════════════════════════════════════════════════════════════════
COLUNAS_NECESSARIAS = {

    # ── Dimensões cadastrais ──────────────────────────────────────────────────
    "TB_PESSOA": {
        "ID_PESSOA": "chave do cliente — unidade de análise do modelo",
        "TP_TIPO": "F/J/E → feature categórica PERFIL",
        "NM_PESSOA": "nome para o lookup de produção (NOME_CLIENTE)",
    },
    "TB_CLIENTE": {
        "ID_CLIENTE": "chave usada pelo comodato",
        "ID_PESSOA": "ponte ID_CLIENTE ↔ ID_PESSOA (as duas trilhas usam chaves diferentes)",
        "DS_FANTASIA": "nome fantasia — ranking de clientes na EDA 3.1",
        "ID_TIPO_ESTABELECIMENTO": "FK do segmento",
        "ID_FORMA_PAGAMENTO_1": "FK da forma de pagamento → feature PAGAMENTO",
    },
    "TB_CLIENTE_ENDERECO": {
        "ID_CLIENTE": "chave de junção",
        "DS_CIDADE": "→ feature categórica CIDADE (agrupada por agrupar_cidades)",
    },
    "TB_FORMA_PAGTO": {
        "ID_FORMA_PAGTO": "chave de junção",
        "DS_FORMA_PAGTO": "descrição → feature PAGAMENTO",
    },
    "TB_TIPO_ESTABELECIMENTO": {
        "ID_TIPO_ESTABELECIMENTO": "chave de junção",
        "DS_TIPO_ESTABELECIMENTO": "segmento do estabelecimento — EDA 3.1",
    },
    "TB_PRODUTO": {
        "ID_PRODUTO": "chave de junção",
        "DS_PRODUTO": "usado pelo REGEX_CORE para isolar chopp/chopeira do resto",
    },

    # ── Fato: itens de pedido (base da view score_vendas) ─────────────────────
    "TB_PEDIDO_ITEM": {
        "ID_PEDIDO": "chave da transação",
        "ID_PRODUTO": "filtro core business + produto favorito",
        "QTD_VENDA": "volume consumido — produto favorito na EDA 3.1",
        "VL_FINANCEIRO": "receita do item → TOTAL_GASTO e TICKET_MEDIO",
    },
    # DT_PEDIDO e ID_PESSOA não vivem no item, e sim no cabeçalho do pedido.
    # Por isso TB_PEDIDO entra: é o que dá recência/frequência (RFM) ao modelo.
    "TB_PEDIDO": {
        "ID_PEDIDO": "chave de junção com o item",
        "ID_PESSOA": "dono do pedido",
        "DT_PEDIDO": "→ PRIMEIRA_COMPRA, ULTIMA_COMPRA, DIAS_DESDE_* e data_corte",
        "DT_ACERTO": "fechamento do pedido — ciclo de vida (EDA)",
        "ID_STATUS": "filtro de pedidos consolidados",
    },

    # ── Fato: financeiro (base da view score_financeiro) ──────────────────────
    # O dump NÃO tem 'TB_FINANCEIRO'. A informação de parcela vive em
    # TB_CONTAS_A_RECEBER_PARCELA e o vínculo com pedido/pessoa em
    # TB_CONTAS_A_RECEBER. A versão anterior deste notebook procurava
    # TB_FINANCEIRO, não achava, e gravava um score_financeiro.csv VAZIO —
    # o que zerava silenciosamente metade da variável-alvo.
    "TB_CONTAS_A_RECEBER": {
        "ID_CONTAS_A_RECEBER": "chave do título",
        "ID_PESSOA": "cliente devedor — chave da trilha financeira",
        "ID_PEDIDO": "pedido que originou a cobrança",
    },
    "TB_CONTAS_A_RECEBER_PARCELA": {
        "ID_CONTAS_A_RECEBER": "FK do título",
        "NR_PARCELA": "identifica o parcelamento",
        "DT_VENCIMENTO": "vencimento — base do cálculo de atraso",
        "DT_RECEBIMENTO": "data de recebimento (fallback de DT_BAIXA)",
        "DT_BAIXA": "liquidação efetiva — usada por calc_atraso_financeiro()",
        "VL_PARCELA": "valor em atraso — série histórica 3.5",
        "VL_RECEBIDO": "detecta pagamento parcial",
        "TP_BAIXA": "auditoria de como o título foi liquidado",
    },

    # ── Fato: comodato (base da view score_comodato) ──────────────────────────
    "TB_COMODATO": {
        "ID_COMODATO": "chave do contrato → TOTAL_COMODATOS",
        "ID_CLIENTE": "cliente — chave da trilha de comodato",
        "ID_PEDIDO": "pedido vinculado",
        "DT_EMPRESTIMO": "início da cessão",
        "DT_VENCIMENTO": "prazo de devolução — base do atraso",
        "DT_RECOLHE": "devolução efetiva — usada por calc_atraso_comodato()",
        "ID_STATUS": "situação do contrato",
    },
    # Qual equipamento e quantos: QTD_PRODUTO alimenta a série de unidades
    # retidas (EDA 3.5) e ID_PRODUTO permite o filtro core business.
    "TB_COMODATO_BEM": {
        "ID_COMODATO": "FK do contrato",
        "ID_PRODUTO": "equipamento cedido — filtro core business",
        "QTD_PRODUTO": "unidades cedidas — série histórica 3.5",
    },
}

# Tabelas a extrair = chaves do contrato. Uma fonte só de verdade.
TABELAS_ALVO = list(COLUNAS_NECESSARIAS)

_total_cols = sum(len(v) for v in COLUNAS_NECESSARIAS.values())
print("⚙️  Ambiente configurado!")
print(f"   ├─ Tabelas no contrato : {len(TABELAS_ALVO)}")
print(f"   ├─ Colunas necessárias : {_total_cols}")
print(f"   └─ Saída               : {OUTPUT_DIR}/")
print("\n   💡 O contrato acima é a ligação com Chopp_Risco_EDA_ML_v1.3.ipynb:")
print("      feature nova lá ⇒ coluna de origem aqui.")

⚙️  Ambiente configurado!
   ├─ Tabelas no contrato : 12
   ├─ Colunas necessárias : 46
   └─ Saída               : C:\_Pessoas\Bruno.Araujo\CDN_ProjetoIntegrador6\Desenvolvimento\data/

   💡 O contrato acima é a ligação com Chopp_Risco_EDA_ML_v1.3.ipynb:
      feature nova lá ⇒ coluna de origem aqui.


## 🛠️ Etapa 1: Parser com Projeção de Colunas

O backup é volumoso e traz comandos estruturais de servidores Windows incompatíveis com o ambiente de análise. A rotina abaixo:

1. Varre o arquivo linha a linha, tratando apenas instruções `INSERT` das tabelas do contrato.
2. Resolve, na primeira ocorrência de cada tabela, o mapa **nome de coluna → posição** e guarda apenas os índices que interessam.
3. Neutraliza funções `CAST(...)` do SQL Server e alinha linhas com contagem divergente de colunas por causa de `NULL`.
4. Materializa em memória **somente as colunas projetadas**.

> A projeção durante o parse não é otimização cosmética: em `TB_PEDIDO_ITEM` (24.399 linhas × 200 colunas) é a diferença entre carregar 4,9 milhões de células e carregar 98 mil.

In [5]:
# ==============================================================================
# 2. PARSER DO DUMP SQL — PROJEÇÃO DE COLUNAS NA LEITURA
# ==============================================================================
# A projeção acontece DURANTE o parse, não depois: as ~630 colunas fora do
# contrato são descartadas linha a linha e nunca chegam à memória. Com
# TB_PEDIDO_ITEM (24.399 linhas × 200 colunas) isso é a diferença entre
# carregar 4,9 milhões de células e carregar 98 mil.
# ==============================================================================
print("⏳ [ETAPA 1/3] Lendo o dump SQL e projetando apenas as colunas do contrato...\n")

padrao_insert = re.compile(
    r"INSERT\s+\[dbo\]\.\[([A-Z0-9_]+)\]\s*\((.*?)\)\s*VALUES\s*\((.*)\);?",
    re.IGNORECASE,
)

# CAST(...) do SQL Server — o tipo pode vir parametrizado: Decimal(18, 2).
# `[A-Za-z]+(?:\([^)]*\))?` cobre tanto DateTime quanto Decimal(18, 2).
_RE_CAST_TEXTO = re.compile(
    r"CAST\(\s*N?'([^']*)'\s+AS\s+[A-Za-z]+(?:\([^)]*\))?\s*\)", re.IGNORECASE
)
_RE_CAST_NUM = re.compile(
    r"CAST\(\s*([^'(),]*?)\s+AS\s+[A-Za-z]+(?:\([^)]*\))?\s*\)", re.IGNORECASE
)

def _limpar(valor: str):
    """
    Normaliza um valor bruto vindo do dump.

    O SQL Server prefixa TODO literal de texto Unicode com N: N'CHOPP 50 LTS'.
    O csv.reader consome as aspas externas mas devolve o N e as internas, então
    o valor chega como o texto  N'CHOPP 50 LTS'  em vez de  CHOPP 50 LTS.

    Efeito prático se não tratado: TP_TIPO vira "N'F'" e o mapa {F, J, E} não
    casa — PERFIL fica "Não Informado" para 100% dos clientes. O mesmo vale para
    CIDADE e PAGAMENTO. São 3 das 12 features do modelo, silenciosamente
    degradadas a uma categoria constante (o OneHotEncoder não reclama disso).
    """
    v = valor.strip()
    if v.upper() == "NULL":
        return None

    # Remove o prefixo N de literal Unicode, com ou sem a aspa de fechamento.
    # A aspa final pode faltar quando o próprio texto contém uma vírgula: o
    # csv.reader corta o campo ali e a sobra vai para o campo seguinte. Ex.:
    # N'CASAS NOTURNAS, BOATES'  →  campo 1: "N'CASAS NOTURNAS"
    # Preferimos o texto sem a aspa a manter o N' grudado no valor.
    if v[:2].upper() == "N'":
        v = v[2:]
    elif v.startswith("'"):
        v = v[1:]
    if v.endswith("'") and not v.endswith("''"):
        v = v[:-1]

    # '' é o escape de aspa simples dentro de string no SQL Server
    return v.replace("''", "'").strip()


dados_memoria = {t: [] for t in TABELAS_ALVO}
colunas_arquivo = {}    # colunas como vêm no dump
indices_projecao = {}   # posições a manter, por tabela
linhas_descartadas = {t: 0 for t in TABELAS_ALVO}

def _registros_insert(caminho):
    """
    Gera um registro INSERT completo por iteração.

    Ler o arquivo linha a linha NÃO basta: campos de texto do ERP contêm
    quebras de linha literais (observações como 'CLIENTE NAO CONFIRMOU\\n...'),
    e nesses casos um único INSERT ocupa várias linhas do dump. Cortar na
    quebra trunca o registro no meio — e o efeito visível é um CAST sem
    parêntese de fechamento, que depois vira data inválida no pandas.

    Acumulamos linhas até o número de aspas simples ficar par (string fechada)
    E a linha terminar em ')' — só então o registro está completo.
    """
    buffer = ""
    with open(caminho, "r", encoding="utf-8", errors="ignore") as f:
        for linha in f:
            if buffer:
                buffer += linha
            elif linha.startswith("INSERT"):
                buffer = linha
            else:
                continue

            if buffer.count("'") % 2 == 0 and buffer.rstrip().rstrip(";").endswith(")"):
                yield buffer
                buffer = ""
    if buffer:
        yield buffer


for linha in _registros_insert(SQL_FILE):
    # A quebra de linha que estava DENTRO de um campo de texto vira espaço:
    # o valor continua legível e o registro passa a caber numa linha lógica,
    # que é o que csv.reader espera.
    linha = linha.replace("\r", " ").replace("\n", " ")

    match = padrao_insert.search(linha)
    if not match:
        continue

    tabela = match.group(1).upper()
    if tabela not in COLUNAS_NECESSARIAS:
        continue

    # Na primeira ocorrência, resolve o mapa nome→posição desta tabela
    if tabela not in colunas_arquivo:
        cols_dump = [c.strip().strip("[]") for c in match.group(2).split(",")]
        colunas_arquivo[tabela] = cols_dump
        desejadas = COLUNAS_NECESSARIAS[tabela]
        indices_projecao[tabela] = [
            (cols_dump.index(c), c) for c in desejadas if c in cols_dump
        ]
        ausentes = [c for c in desejadas if c not in cols_dump]
        if ausentes:
            print(f"   ⚠️  {tabela}: colunas do contrato ausentes no dump → {ausentes}")

    # Neutraliza CAST(...) do SQL Server antes de passar ao csv.reader.
    #
    # Dois padrões, porque o tipo pode ser PARAMETRIZADO:
    #   CAST(N'2024-04-25T13:51:55.320' AS DateTime)  → literal com aspas
    #   CAST(12.5 AS Decimal(18, 2))                  → literal numérico
    # O `(?:\([^)]*\))?` é o ponto crítico: sem ele, um `[^)]+` para no
    # primeiro `)` interno de Decimal(18, 2) e deixa um `)` órfão no meio
    # da linha. O csv.reader então desalinha TODOS os campos seguintes e a
    # data chega ao pandas como "CAST(N'...' AS DateTime" — corrompendo a
    # linha inteira sem lançar erro nenhum aqui.
    val_raw = _RE_CAST_TEXTO.sub(r"'\1'", match.group(3))
    val_raw = _RE_CAST_NUM.sub(r"\1", val_raw)

    leitor = csv.reader([val_raw], delimiter=",", quotechar="'", skipinitialspace=True)

    n_cols = len(colunas_arquivo[tabela])
    for row in leitor:
        valores = [_limpar(v) for v in row]

        # Alinhamento defensivo: o dump pode trazer linhas curtas por NULLs
        if len(valores) < n_cols:
            valores.extend([None] * (n_cols - len(valores)))
        elif len(valores) > n_cols:
            valores = valores[:n_cols]

        # ── PROJEÇÃO: só as posições do contrato entram na memória ────────
        try:
            dados_memoria[tabela].append([valores[i] for i, _ in indices_projecao[tabela]])
        except IndexError:
            linhas_descartadas[tabela] += 1

# ── Materialização em DataFrames ─────────────────────────────────────────────
tabelas_db = {}
print(f"{'Tabela':<32} {'Linhas':>9} {'Cols dump':>10} {'Cols mantidas':>14}")
print("-" * 70)

total_orig = total_mantido = 0
for tabela in TABELAS_ALVO:
    if not dados_memoria[tabela]:
        print(f"{tabela:<32} {'—':>9}  (nenhum INSERT encontrado no dump)")
        continue

    nomes = [nome for _, nome in indices_projecao[tabela]]
    df = pd.DataFrame(dados_memoria[tabela], columns=nomes)
    tabelas_db[tabela] = df

    n_dump = len(colunas_arquivo[tabela])
    total_orig += n_dump
    total_mantido += len(nomes)
    print(f"{tabela:<32} {len(df):>9,} {n_dump:>10} {len(nomes):>14}")

    if linhas_descartadas[tabela]:
        print(f"{'':<32} ⚠️  {linhas_descartadas[tabela]} linha(s) malformada(s) descartada(s)")

print("-" * 70)
print(f"{'TOTAL':<32} {'':>9} {total_orig:>10} {total_mantido:>14}")
if total_orig:
    print(f"\n   📉 Redução de colunas: {total_orig} → {total_mantido} "
          f"({100 - total_mantido / total_orig * 100:.1f}% descartado)")

# ── Verificação do contrato ──────────────────────────────────────────────────
# Falha cedo e alto: uma tabela ausente aqui vira coluna faltando na Fase 4 do
# notebook de ML, dezenas de células depois, com mensagem muito menos clara.
_faltando = [t for t in TABELAS_ALVO if t not in tabelas_db]
if _faltando:
    raise RuntimeError(
        f"Tabelas do contrato não encontradas no dump: {_faltando}. "
        f"Verifique se SQL_FILE aponta para o backup completo."
    )

# ── Detector de resíduo de CAST ──────────────────────────────────────────────
# Um CAST mal desfeito não gera exceção aqui: ele vira o texto
# "CAST(N'2024-04-25...' AS DateTime" dentro de uma célula, e só estoura muitas
# células depois, em pd.to_datetime, com mensagem que não aponta para a causa.
# Barato de checar, caro de diagnosticar depois.
_residuos = {}
for tabela, df in tabelas_db.items():
    n = 0
    for col in df.columns:
        if df[col].dtype == "object":
            n += int(df[col].astype(str).str.contains("CAST(", regex=False, na=False).sum())
    if n:
        _residuos[tabela] = n

if _residuos:
    raise RuntimeError(
        f"Resíduo de CAST() encontrado após o parse: {_residuos}. "
        f"As linhas afetadas estão com os campos desalinhados. "
        f"Verifique _RE_CAST_TEXTO/_RE_CAST_NUM — provavelmente há um tipo "
        f"parametrizado novo no dump que os padrões não cobrem."
    )

# ── Detector de prefixo N' residual ──────────────────────────────────────────
# Mesma lógica do detector de CAST: um N' que sobrou não quebra nada aqui,
# só transforma uma feature categórica em constante lá na frente.
_prefixo_n = {}
for tabela, df in tabelas_db.items():
    n = 0
    for col in df.columns:
        if df[col].dtype == "object":
            n += int(df[col].astype(str).str.match(r"^N'").sum())
    if n:
        _prefixo_n[tabela] = n

if _prefixo_n:
    raise RuntimeError(
        f"Prefixo N' de literal Unicode não removido: {_prefixo_n}. "
        f"Colunas de texto ficariam com o valor errado (ex.: TP_TIPO=\"N'F'\" "
        f"em vez de \"F\"), degradando features categóricas a constantes. "
        f"Verifique _limpar()."
    )

print("\n   ✅ Todas as tabelas do contrato foram extraídas.")
print("      Sem resíduo de CAST() e sem prefixo N' de literal Unicode.")

⏳ [ETAPA 1/3] Lendo o dump SQL e projetando apenas as colunas do contrato...

Tabela                              Linhas  Cols dump  Cols mantidas
----------------------------------------------------------------------
TB_PESSOA                            1,473         27              3
TB_CLIENTE                           1,432         65              5
TB_CLIENTE_ENDERECO                  1,418         28              2
TB_FORMA_PAGTO                          10         29              2
TB_TIPO_ESTABELECIMENTO                 90         10              2
TB_PRODUTO                             127         81              2
TB_PEDIDO_ITEM                      24,399        200              4
TB_PEDIDO                           11,680        120              5
TB_CONTAS_A_RECEBER                  5,885         27              3
TB_CONTAS_A_RECEBER_PARCELA          6,831         49              8
TB_COMODATO                          2,735         21              7
TB_COMODATO_BEM        

## 🔗 Etapas 2 e 3: Consolidação em Arquivo Único e Auditoria

As três trilhas transacionais são remontadas a partir das tabelas normalizadas e consolidadas em **um único arquivo**: `dataset_consolidado.csv`, com **1 linha por cliente** (`ID_PESSOA`). É a base do projeto — tanto a EDA quanto a modelagem leem dele.

| Trilha | Junção | Por que a junção é necessária |
| :--- | :--- | :--- |
| vendas | `TB_PEDIDO_ITEM` ⋈ `TB_PEDIDO` | o item não tem `ID_PESSOA` nem `DT_PEDIDO` — sem o cabeçalho não há RFM |
| financeiro | `TB_CONTAS_A_RECEBER_PARCELA` ⋈ `TB_CONTAS_A_RECEBER` | a parcela tem as datas; o título tem o cliente e o pedido |
| comodato | `TB_COMODATO` ⋈ `TB_COMODATO_BEM` | o contrato tem cliente e prazos; o bem diz qual equipamento e quantos |

### ⚠️ Por que não encadear as três num join só

Um pedido tem **N itens × M parcelas × K bens de comodato**. Encadeá-las por `ID_PEDIDO` multiplica as linhas — medido neste dump: **24.399 → 43.562 (1,8×)** — e faz o mesmo `VL_FINANCEIRO` ser contado várias vezes, **inflando o faturamento** sem que nada indique o erro. Era o defeito do `dataset_consolidado.csv` de 300 colunas da versão anterior.

A consolidação correta **agrega cada trilha na sua própria granularidade** — faturamento sobre itens, atrasos sobre parcelas, retenção sobre contratos — e só então une os resultados pela chave do cliente. A auditoria confere que a soma do faturamento por cliente bate **ao centavo** com a soma das linhas de origem; é essa checagem que teria detectado a inflação antiga.

### 📋 O que o arquivo contém (38 colunas)

| Grupo | Colunas |
| :--- | :--- |
| **Identificação** | `ID_PESSOA`, `NOME_CLIENTE`, `DS_FANTASIA` |
| **Cadastro** (features categóricas) | `PERFIL`, `CIDADE`, `PAGAMENTO`, `SEGMENTO` |
| **RFM** | `PRIMEIRA_COMPRA`, `ULTIMA_COMPRA`, `MES_ULTIMA_COMPRA`, `DIAS_DESDE_*`, `FREQUENCIA_COMPRAS`, `TOTAL_ITENS`, `QTD_TOTAL_VENDIDA`, `TOTAL_GASTO`, `TICKET_MEDIO`, `PRODUTO_FAVORITO` |
| **Risco financeiro** | `TOTAL_PARCELAS`, `PARCELAS_ATRASADAS`, `TAXA_ATRASO_PAGAMENTO`, `MEDIA/MAX_DIAS_ATRASO_PAG`, `VALOR_TOTAL_PARCELAS`, `AGING_PAGAMENTO` |
| **Risco de comodato** | `TOTAL_COMODATOS`, `COMODATOS_ATRASADOS`, `TAXA_ATRASO_COMODATO`, `MEDIA/MAX_DIAS_ATRASO_COM`, `QTD_EQUIPAMENTOS`, `AGING_COMODATO` |
| **Apoio à EDA** | `RISCO_FINANCEIRO`, `RISCO_COMODATO`, `PERFIL_RISCO`, `TEM_VENDAS`, `TEM_FINANCEIRO`, `TEM_COMODATO` |

As colunas de **aging** usam o atraso **máximo**, não o médio: um cliente com um atraso de 45 dias é caso de "+30 Dias", ainda que a média o diluísse numa faixa branda. As flags `TEM_*` distinguem "não tem comodato" de "comodato não encontrado" — leituras muito diferentes que um zero sozinho confunde.

### 🔑 Três correções de integridade

**1. `TB_FINANCEIRO` não existe neste dump.** A versão anterior a procurava, não achava, e gravava um `score_financeiro.csv` **vazio com colunas inventadas**. O notebook de ML lia sem erro e calculava `TAXA_ATRASO_PAGAMENTO = 0` para todos — zerando metade da variável-alvo. Os dados reais estão em `TB_CONTAS_A_RECEBER_PARCELA` ⋈ `TB_CONTAS_A_RECEBER`.

**2. `TB_COMODATO.ID_CLIENTE` contém `ID_PESSOA`.** A coluna tem nome de uma chave e valores de outra. Verificação independente, usando o dono do pedido como fonte da verdade:

| Interpretação | Contratos coerentes com o dono do pedido |
| :--- | :--- |
| traduzir via `TB_CLIENTE` (o que se fazia) | 72 / 8.180 — **0,9%** |
| tratar o valor como `ID_PESSOA` | 8.180 / 8.180 — **100%** |

Como os dois espaços de ID se sobrepõem (`ID_CLIENTE` 5 ↔ `ID_PESSOA` 9, e existe uma pessoa 5), a tradução não apenas perdia contratos: atribuía inadimplência de comodato **ao cliente errado**.

**3. Literais Unicode chegavam com o prefixo `N`.** O SQL Server escreve `N'CHOPP 50 LTS'`, e o valor chegava ao DataFrame como o texto `N'F'` em vez de `F`. Efeito: `PERFIL`, `CIDADE` e `PAGAMENTO` — **3 das 12 features do modelo** — viravam a constante "Não Informado", e o `OneHotEncoder` não reclama de uma categoria só. Após a correção: PERFIL com 3 valores, CIDADE com 5, PAGAMENTO com 6.

> As três falhas eram **silenciosas** — nenhuma lançava exceção. Por isso a célula agora falha explicitamente diante de resíduo de `CAST()`, prefixo `N'`, view vazia, faturamento não conservado ou chave duplicada.

In [6]:
# ==============================================================================
# 3. CONSOLIDAÇÃO EM ARQUIVO ÚNICO — dataset_consolidado.csv
# ==============================================================================
#  Saída: UM arquivo, 1 linha por CLIENTE (ID_PESSOA). É a base do projeto:
#  tanto a EDA quanto a modelagem leem dele.
#
#  Por que não encadear as três trilhas num join só: um pedido tem N itens,
#  M parcelas e K bens de comodato. Encadeá-las por ID_PEDIDO multiplica as
#  linhas — medido neste dump: 24.399 → 43.562 (1,8×) — e faz o mesmo
#  VL_FINANCEIRO ser contado várias vezes, inflando o faturamento em 1,1% sem
#  que nada indique o erro. Foi o que aconteceu no dataset_consolidado de 300
#  colunas da versão anterior.
#
#  A consolidação correta agrega CADA trilha na sua própria granularidade
#  (faturamento sobre itens, atrasos sobre parcelas, retenção sobre contratos)
#  e só então une os RESULTADOS, pela chave do cliente. A auditoria ao final
#  confere que o faturamento por cliente bate ao centavo com o de origem.
# ==============================================================================
print("\n🔗 [ETAPA 2/3] Montando as views transacionais...\n")


def _num(serie):
    """Converte para numérico preservando NaN — os CSVs vêm todos como string."""
    return pd.to_numeric(serie, errors="coerce")


# ── VIEW 1: vendas (item de pedido ⋈ cabeçalho) ───────────────────────────────
# O item não tem ID_PESSOA nem DT_PEDIDO: sem o cabeçalho não há RFM.
df_vendas = tabelas_db["TB_PEDIDO_ITEM"].merge(
    tabelas_db["TB_PEDIDO"][["ID_PEDIDO", "ID_PESSOA", "DT_PEDIDO", "DT_ACERTO", "ID_STATUS"]],
    on="ID_PEDIDO",
    how="inner",   # item órfão de pedido não tem dono nem data
)

# ── VIEW 2: financeiro (parcela ⋈ título) ─────────────────────────────────────
# A parcela tem as datas; o título tem ID_PESSOA e ID_PEDIDO.
df_financeiro = tabelas_db["TB_CONTAS_A_RECEBER_PARCELA"].merge(
    tabelas_db["TB_CONTAS_A_RECEBER"][["ID_CONTAS_A_RECEBER", "ID_PESSOA", "ID_PEDIDO"]],
    on="ID_CONTAS_A_RECEBER",
    how="inner",
)

# ── VIEW 3: comodato (contrato ⋈ bem) ─────────────────────────────────────────
df_comodato = tabelas_db["TB_COMODATO"].merge(
    tabelas_db["TB_COMODATO_BEM"][["ID_COMODATO", "ID_PRODUTO", "QTD_PRODUTO"]],
    on="ID_COMODATO",
    how="left",    # contrato sem bem registrado ainda é comodato em aberto
)

# ══════════════════════════════════════════════════════════════════════════════
#  ⚠️ CORREÇÃO DE CHAVE — TB_COMODATO.ID_CLIENTE contém ID_PESSOA
#
#  A coluna se chama ID_CLIENTE, mas os valores gravados nela são ID_PESSOA.
#  Verificação independente, usando o dono do pedido como fonte da verdade
#  (ID_PEDIDO do comodato → TB_PEDIDO.ID_PESSOA):
#
#     tratar como ID_CLIENTE e traduzir via TB_CLIENTE →   72/8.180 batem (0,9%)
#     tratar o valor como ID_PESSOA diretamente        → 8.180/8.180 batem (100%)
#
#  Traduzir via TB_CLIENTE não apenas perdia 46% dos contratos: dos que
#  "resolvia", quase todos apontavam para a PESSOA ERRADA — porque os IDs dos
#  dois espaços se sobrepõem (ID_CLIENTE 5 ↔ ID_PESSOA 9, e existe uma pessoa 5).
#  O efeito era inadimplência de comodato atribuída a quem não a tinha.
#
#  Renomeamos para ID_PESSOA e validamos contra o pedido logo abaixo.
# ══════════════════════════════════════════════════════════════════════════════
df_comodato = df_comodato.rename(columns={"ID_CLIENTE": "ID_PESSOA"})

_dono_pedido = df_vendas[["ID_PEDIDO", "ID_PESSOA"]].drop_duplicates("ID_PEDIDO")
_chk = df_comodato.merge(
    _dono_pedido.rename(columns={"ID_PESSOA": "_DONO_PEDIDO"}), on="ID_PEDIDO", how="inner"
)
if len(_chk):
    _coerentes = (_num(_chk["ID_PESSOA"]) == _num(_chk["_DONO_PEDIDO"])).sum()
    _pct = _coerentes / len(_chk) * 100
    print(f"   🔑 Validação da chave do comodato: {_coerentes:,}/{len(_chk):,} "
          f"contratos ({_pct:.1f}%) coincidem com o dono do pedido.")
    if _pct < 95:
        print(f"   ⚠️  Coerência abaixo do esperado — reveja a semântica de "
              f"TB_COMODATO.ID_CLIENTE neste dump.")


# ══════════════════════════════════════════════════════════════════════════════
#  ARQUIVO 1 — dataset_consolidado.csv  ·  1 LINHA POR CLIENTE
#
#  Cada trilha é agregada na sua própria granularidade ANTES da união. É isso
#  que impede a duplicação: o faturamento é somado sobre os itens, as parcelas
#  sobre as parcelas, os comodatos sobre os contratos — e só os RESULTADOS se
#  encontram, um por cliente.
# ══════════════════════════════════════════════════════════════════════════════
print("\n🧮 [ETAPA 3/3] Consolidando por cliente...\n")

DATA_CORTE = pd.to_datetime(df_vendas["DT_PEDIDO"], errors="coerce").max()
print(f"   📅 Data de corte (último pedido): {DATA_CORTE:%d/%m/%Y}")

# ── Trilha A: vendas → RFM ────────────────────────────────────────────────────
_v = df_vendas.copy()
_v["DT_PEDIDO"] = pd.to_datetime(_v["DT_PEDIDO"], errors="coerce")
_v["VL_FINANCEIRO"] = _num(_v["VL_FINANCEIRO"])
_v["QTD_VENDA"] = _num(_v["QTD_VENDA"])

agg_vendas = (
    _v.groupby("ID_PESSOA")
    .agg(
        PRIMEIRA_COMPRA=("DT_PEDIDO", "min"),
        ULTIMA_COMPRA=("DT_PEDIDO", "max"),
        FREQUENCIA_COMPRAS=("ID_PEDIDO", "nunique"),
        TOTAL_ITENS=("ID_PEDIDO", "count"),
        TOTAL_GASTO=("VL_FINANCEIRO", "sum"),
        QTD_TOTAL_VENDIDA=("QTD_VENDA", "sum"),
    )
    .reset_index()
)
agg_vendas["TICKET_MEDIO"] = agg_vendas["TOTAL_GASTO"] / agg_vendas["FREQUENCIA_COMPRAS"]
agg_vendas["DIAS_DESDE_PRIMEIRA_COMPRA"] = (DATA_CORTE - agg_vendas["PRIMEIRA_COMPRA"]).dt.days
agg_vendas["DIAS_DESDE_ULTIMA_COMPRA"] = (DATA_CORTE - agg_vendas["ULTIMA_COMPRA"]).dt.days

# Produto favorito: item mais consumido em volume. A EDA 3.1 o exibe no
# ranking de clientes — sem ele seria preciso voltar ao grão transacional.
_prod = tabelas_db["TB_PRODUTO"][["ID_PRODUTO", "DS_PRODUTO"]]
_favorito = (
    _v.groupby(["ID_PESSOA", "ID_PRODUTO"])["QTD_VENDA"].sum().reset_index()
    .sort_values(["ID_PESSOA", "QTD_VENDA"], ascending=[True, False])
    .drop_duplicates("ID_PESSOA")
    .merge(_prod, on="ID_PRODUTO", how="left")
    [["ID_PESSOA", "DS_PRODUTO"]]
    .rename(columns={"DS_PRODUTO": "PRODUTO_FAVORITO"})
)
agg_vendas = agg_vendas.merge(_favorito, on="ID_PESSOA", how="left")

# ── Marcação de Core Business ─────────────────────────────────────────────────
# Quais clientes consomem chopp/chopeira. A marcação precisa ser feita AQUI,
# no grão transacional, porque depende de ID_PRODUTO — informação que não
# sobrevive à agregação por cliente. Vai como flag para o arquivo final, e a
# Fase 4 do notebook de ML filtra por ela.
REGEX_CORE = r"CHOPP|CHOPEIRA|BARRIL|CILINDRO|VÁLVULA|VALVULA|EXTRATORA|GÁS|GAS|CIL"
REGEX_RUIDO = r"COPO|DESCARTÁVEL|DESCARTAVEL|GELO|ÁGUA|AGUA|REFRIGERANTE|SUCO"

_prod_core = tabelas_db["TB_PRODUTO"].copy()
_prod_core["DS_PRODUTO"] = _prod_core["DS_PRODUTO"].fillna("").str.upper()
_ids_core = _prod_core[
    _prod_core["DS_PRODUTO"].str.contains(REGEX_CORE, regex=True)
    & ~_prod_core["DS_PRODUTO"].str.contains(REGEX_RUIDO, regex=True)
]["ID_PRODUTO"].unique()

_clientes_core = set(_v[_v["ID_PRODUTO"].isin(_ids_core)]["ID_PESSOA"].dropna())
_fat_core = _v[_v["ID_PRODUTO"].isin(_ids_core)]["VL_FINANCEIRO"].sum()

print(f"   🍺 Core business: {len(_ids_core)} produtos, {len(_clientes_core):,} clientes, "
      f"R$ {_fat_core:,.2f}")


# ── Trilha B: financeiro → atraso de pagamento ────────────────────────────────
# Mesma regra de calc_atraso_financeiro() do notebook de ML: atrasou se pagou
# depois do vencimento OU não pagou e o vencimento já passou.
_f = df_financeiro.copy()
_f["DT_VENCIMENTO"] = pd.to_datetime(_f["DT_VENCIMENTO"], errors="coerce")
_col_pgto = "DT_BAIXA" if "DT_BAIXA" in _f.columns else "DT_RECEBIMENTO"
_f[_col_pgto] = pd.to_datetime(_f[_col_pgto], errors="coerce")
_f["VL_PARCELA"] = _num(_f["VL_PARCELA"])

_f["DIAS_ATRASO_PAG"] = (
    _f[_col_pgto].fillna(DATA_CORTE) - _f["DT_VENCIMENTO"]
).dt.days.clip(lower=0)
_f["ATRASOU_PAGAMENTO"] = (
    (_f[_col_pgto] > _f["DT_VENCIMENTO"])
    | (_f[_col_pgto].isna() & (_f["DT_VENCIMENTO"] < DATA_CORTE))
).astype(int)

agg_fin = (
    _f.groupby("ID_PESSOA")
    .agg(
        TOTAL_PARCELAS=("ID_CONTAS_A_RECEBER", "count"),
        PARCELAS_ATRASADAS=("ATRASOU_PAGAMENTO", "sum"),
        MEDIA_DIAS_ATRASO_PAG=("DIAS_ATRASO_PAG", "mean"),
        MAX_DIAS_ATRASO_PAG=("DIAS_ATRASO_PAG", "max"),
        VALOR_TOTAL_PARCELAS=("VL_PARCELA", "sum"),
    )
    .reset_index()
)
agg_fin["TAXA_ATRASO_PAGAMENTO"] = (
    agg_fin["PARCELAS_ATRASADAS"] / agg_fin["TOTAL_PARCELAS"].replace(0, 1)
)

# ── Trilha C: comodato → atraso de devolução ──────────────────────────────────
# Atenção à granularidade: o contrato é a unidade de risco, mas a view tem uma
# linha por BEM. Deduplicamos por ID_COMODATO antes de contar contratos, senão
# um comodato com 3 chopeiras contaria como 3 atrasos.
_c = df_comodato.copy()
_c["DT_VENCIMENTO"] = pd.to_datetime(_c["DT_VENCIMENTO"], errors="coerce")
_c["DT_RECOLHE"] = pd.to_datetime(_c["DT_RECOLHE"], errors="coerce")
_c["QTD_PRODUTO"] = _num(_c["QTD_PRODUTO"])

_c["DIAS_ATRASO_COM"] = (
    _c["DT_RECOLHE"].fillna(DATA_CORTE) - _c["DT_VENCIMENTO"]
).dt.days.clip(lower=0)
_c["ATRASOU_COMODATO"] = (
    (_c["DT_RECOLHE"] > _c["DT_VENCIMENTO"])
    | (_c["DT_RECOLHE"].isna() & (_c["DT_VENCIMENTO"] < DATA_CORTE))
).astype(int)

# Unidades cedidas somam por bem; o risco conta por contrato.
_qtd_por_cliente = _c.groupby("ID_PESSOA")["QTD_PRODUTO"].sum().rename("QTD_EQUIPAMENTOS")
_contratos = _c.drop_duplicates("ID_COMODATO")

agg_com = (
    _contratos.groupby("ID_PESSOA")
    .agg(
        TOTAL_COMODATOS=("ID_COMODATO", "nunique"),
        COMODATOS_ATRASADOS=("ATRASOU_COMODATO", "sum"),
        MEDIA_DIAS_ATRASO_COM=("DIAS_ATRASO_COM", "mean"),
        MAX_DIAS_ATRASO_COM=("DIAS_ATRASO_COM", "max"),
    )
    .reset_index()
    .merge(_qtd_por_cliente, on="ID_PESSOA", how="left")
)
agg_com["TAXA_ATRASO_COMODATO"] = (
    agg_com["COMODATOS_ATRASADOS"] / agg_com["TOTAL_COMODATOS"].replace(0, 1)
)

# ── Dimensões cadastrais (1 linha por cliente) ────────────────────────────────
MAPA_PESSOA = {"F": "Física", "J": "Jurídica", "E": "Estrangeiro"}

dim = tabelas_db["TB_PESSOA"][["ID_PESSOA", "TP_TIPO", "NM_PESSOA"]].drop_duplicates("ID_PESSOA").copy()
dim["PERFIL"] = dim["TP_TIPO"].map(MAPA_PESSOA).fillna("Não Informado")
dim = dim.rename(columns={"NM_PESSOA": "NOME_CLIENTE"}).drop(columns=["TP_TIPO"])

_cli = tabelas_db["TB_CLIENTE"][
    ["ID_CLIENTE", "ID_PESSOA", "DS_FANTASIA", "ID_TIPO_ESTABELECIMENTO", "ID_FORMA_PAGAMENTO_1"]
].drop_duplicates("ID_PESSOA")

_cli = _cli.merge(
    tabelas_db["TB_FORMA_PAGTO"][["ID_FORMA_PAGTO", "DS_FORMA_PAGTO"]],
    left_on="ID_FORMA_PAGAMENTO_1", right_on="ID_FORMA_PAGTO", how="left",
).merge(
    tabelas_db["TB_TIPO_ESTABELECIMENTO"][["ID_TIPO_ESTABELECIMENTO", "DS_TIPO_ESTABELECIMENTO"]],
    on="ID_TIPO_ESTABELECIMENTO", how="left",
)
_cli["PAGAMENTO"] = _cli["DS_FORMA_PAGTO"].fillna("OUTROS").str.upper()
_cli["SEGMENTO"] = _cli["DS_TIPO_ESTABELECIMENTO"].fillna("OUTROS")

# Cidade: uma por cliente, já padronizada
_end = tabelas_db["TB_CLIENTE_ENDERECO"][["ID_CLIENTE", "DS_CIDADE"]].drop_duplicates("ID_CLIENTE").copy()


def agrupar_cidades(cidade):
    """Padroniza variações de digitação — mesma regra do notebook de ML."""
    c = str(cidade).upper().strip()
    if any(k in c for k in ["PONTA POR", "SANGA", "SANGRA"]):
        return "PONTA PORÃ"
    if "PEDRO JUAN" in c:
        return "PEDRO JUAN CABALLERO"
    if any(k in c for k in ["AMAMBA", "AMANBA"]):
        return "AMAMBAI"
    if c in ["NAN", "", "NONE", "NÃO PREENCHIDO", "NAO PREENCHIDO"]:
        return "NÃO PREENCHIDO"
    return "OUTRAS CIDADES"


_end["CIDADE"] = _end["DS_CIDADE"].apply(agrupar_cidades)

dim_cliente = (
    _cli[["ID_CLIENTE", "ID_PESSOA", "DS_FANTASIA", "PAGAMENTO", "SEGMENTO"]]
    .merge(_end[["ID_CLIENTE", "CIDADE"]], on="ID_CLIENTE", how="left")
    .drop_duplicates("ID_PESSOA")
)

# ── União: outer, para não perder cliente que só existe numa trilha ──────────
dataset = (
    dim.merge(dim_cliente.drop(columns=["ID_CLIENTE"]), on="ID_PESSOA", how="outer")
    .merge(agg_vendas, on="ID_PESSOA", how="outer")
    .merge(agg_fin, on="ID_PESSOA", how="outer")
    .merge(agg_com, on="ID_PESSOA", how="outer")
)

# Zero é a leitura correta aqui: "nenhuma parcela" é 0 parcelas, não desconhecido.
_cols_zero = [
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "TOTAL_GASTO", "QTD_TOTAL_VENDIDA", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG",
    "VALOR_TOTAL_PARCELAS", "TAXA_ATRASO_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM",
    "QTD_EQUIPAMENTOS", "TAXA_ATRASO_COMODATO",
]
dataset[_cols_zero] = dataset[_cols_zero].fillna(0)

for _col, _val in [("PERFIL", "Não Informado"), ("CIDADE", "NÃO PREENCHIDO"),
                   ("PAGAMENTO", "OUTROS"), ("SEGMENTO", "OUTROS"),
                   ("NOME_CLIENTE", "N/D"), ("DS_FANTASIA", "NÃO INFORMADO")]:
    if _col in dataset.columns:
        dataset[_col] = dataset[_col].fillna(_val)

# Flags de cobertura: dizem em quais trilhas o cliente aparece. Baratas de
# gravar e evitam confundir "não tem comodato" com "comodato não encontrado".
dataset["TEM_VENDAS"] = (dataset["FREQUENCIA_COMPRAS"] > 0).astype(int)
# Flag de core business: marcada no grão transacional (ver acima), porque
# depende de ID_PRODUTO. É por ela que a Fase 4 do notebook de ML seleciona
# o universo de modelagem.
dataset["CORE_BUSINESS"] = dataset["ID_PESSOA"].isin(_clientes_core).astype(int)
dataset["TEM_FINANCEIRO"] = (dataset["TOTAL_PARCELAS"] > 0).astype(int)
dataset["TEM_COMODATO"] = (dataset["TOTAL_COMODATOS"] > 0).astype(int)

# ── Colunas de apoio à EDA, no grão do cliente ───────────────────────────────
# A EDA precisa de aging (faixa de severidade), recorte temporal e um rótulo de
# risco. Todas derivam do que já está agregado — ficam no mesmo arquivo em vez
# de exigir um segundo dataset com o detalhe por parcela.
ORDEM_AGING = ["Sem Atraso", "1-3 Dias", "4-7 Dias", "8-15 Dias",
               "16-20 Dias", "21-30 Dias", "+30 Dias"]


def categorizar_atraso(dias) -> str:
    """Converte dias de atraso em faixa de aging (mesma escala do notebook de ML)."""
    d = pd.to_numeric(dias, errors="coerce")
    if pd.isna(d) or d <= 0:
        return "Sem Atraso"
    if d <= 3:
        return "1-3 Dias"
    if d <= 7:
        return "4-7 Dias"
    if d <= 15:
        return "8-15 Dias"
    if d <= 20:
        return "16-20 Dias"
    if d <= 30:
        return "21-30 Dias"
    return "+30 Dias"


# Aging pela severidade MÁXIMA: um cliente com um atraso de 45 dias é caso de
# "+30 Dias", ainda que a média o diluísse numa faixa branda.
dataset["AGING_PAGAMENTO"] = dataset["MAX_DIAS_ATRASO_PAG"].apply(categorizar_atraso)
dataset["AGING_COMODATO"] = dataset["MAX_DIAS_ATRASO_COM"].apply(categorizar_atraso)

# Recorte temporal no grão do cliente: mês da última compra.
dataset["MES_ULTIMA_COMPRA"] = pd.to_datetime(
    dataset["ULTIMA_COMPRA"], errors="coerce"
).dt.to_period("M").astype(str)

# Rótulo de risco integrado, para os cruzamentos da EDA 3.3 e 3.6.
_LIMITE_EDA = 0.20
dataset["RISCO_FINANCEIRO"] = (dataset["TAXA_ATRASO_PAGAMENTO"] > _LIMITE_EDA).astype(int)
dataset["RISCO_COMODATO"] = (dataset["TAXA_ATRASO_COMODATO"] > _LIMITE_EDA).astype(int)
dataset["PERFIL_RISCO"] = dataset.apply(
    lambda r: (
        "RISCO DUPLO" if r["RISCO_FINANCEIRO"] and r["RISCO_COMODATO"]
        else "SÓ FINANCEIRO" if r["RISCO_FINANCEIRO"]
        else "SÓ COMODATO" if r["RISCO_COMODATO"]
        else "SEM RISCO"
    ),
    axis=1,
)

_ORDEM = [
    "ID_PESSOA", "NOME_CLIENTE", "DS_FANTASIA", "PERFIL", "CIDADE", "PAGAMENTO", "SEGMENTO",
    "PRIMEIRA_COMPRA", "ULTIMA_COMPRA", "MES_ULTIMA_COMPRA",
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA",
    "FREQUENCIA_COMPRAS", "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "PRODUTO_FAVORITO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS", "AGING_PAGAMENTO",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS", "AGING_COMODATO",
    "RISCO_FINANCEIRO", "RISCO_COMODATO", "PERFIL_RISCO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]
dataset = dataset[[c for c in _ORDEM if c in dataset.columns]]

_path_cons = os.path.join(OUTPUT_DIR, "dataset_consolidado.csv")
dataset.to_csv(_path_cons, **ESCRITA_PARAMS)


# ══════════════════════════════════════════════════════════════════════════════
#  AUDITORIA
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 78)
print(f"{'AUDITORIA DE CONSOLIDAÇÃO':^78}")
print("=" * 78)

# Conservação de valor: a soma por cliente tem de bater com a soma das linhas
# de origem. É a checagem que teria detectado a inflação do join em cadeia.
_fat_origem = _v["VL_FINANCEIRO"].sum()
_fat_consol = dataset["TOTAL_GASTO"].sum()
_dif = abs(_fat_origem - _fat_consol)
print(f"\n  Faturamento nas vendas (origem)  : R$ {_fat_origem:>15,.2f}")
print(f"  Faturamento no consolidado       : R$ {_fat_consol:>15,.2f}")
print(f"  Diferença                        : R$ {_dif:>15,.2f}")
if _dif > 0.01:
    raise RuntimeError(
        f"Faturamento não conservado na consolidação (diferença R$ {_dif:,.2f}). "
        f"Provável duplicação de linhas em algum merge."
    )
print("  ✅ Valor conservado — nenhuma duplicação nos merges.")

if dataset["ID_PESSOA"].duplicated().any():
    raise RuntimeError("dataset_consolidado tem ID_PESSOA duplicado — não é 1 linha por cliente.")
print(f"  ✅ Chave única: {len(dataset):,} clientes, um por linha.")

print(f"\n  Cobertura por trilha:")
print(f"    ├─ com vendas     : {int(dataset['TEM_VENDAS'].sum()):>5,} clientes")
print(f"    ├─ com financeiro : {int(dataset['TEM_FINANCEIRO'].sum()):>5,} clientes")
print(f"    ├─ com comodato   : {int(dataset['TEM_COMODATO'].sum()):>5,} clientes")
print(f"    └─ core business  : {int(dataset['CORE_BUSINESS'].sum()):>5,} clientes  (chopp/chopeira)")

# Datas nulas são semanticamente corretas: cliente sem nenhuma venda não tem
# primeira nem última compra. Preencher com 0 mentiria ("comprou hoje"), e com
# -1 inventaria uma data. Ficam NaN e o filtro de core business da Fase 4 os
# remove antes do treino — mas a contagem é reportada para não virar surpresa.
_sem_venda = int((dataset["TEM_VENDAS"] == 0).sum())
if _sem_venda:
    print(f"\n  ℹ️  {_sem_venda} cliente(s) sem venda registrada têm PRIMEIRA_COMPRA,")
    print(f"      ULTIMA_COMPRA e DIAS_DESDE_* nulos — é a leitura correta, não")
    print(f"      um defeito. Eles não entram no treino (não têm produto core).")

print("\n" + "=" * 78)
print("🎉 EXTRAÇÃO E CONSOLIDAÇÃO CONCLUÍDAS")
print("=" * 78)
print(f"  📁 {os.path.basename(_path_cons)}")
print(f"     {dataset.shape[0]:,} clientes × {dataset.shape[1]} colunas "
      f"({os.path.getsize(_path_cons)/1024/1024:.2f} MB)")
print(f"     1 linha por CLIENTE — é o dataset do projeto: EDA e modelo leem daqui.")
print(f"\n  📊 Colunas do dump: {total_mantido} mantidas de {total_orig} "
      f"({100 - total_mantido / total_orig * 100:.1f}% descartado)")
print("=" * 78)


🔗 [ETAPA 2/3] Montando as views transacionais...

   🔑 Validação da chave do comodato: 8,180/8,180 contratos (100.0%) coincidem com o dono do pedido.

🧮 [ETAPA 3/3] Consolidando por cliente...

   📅 Data de corte (último pedido): 25/08/2026
   🍺 Core business: 83 produtos, 1,349 clientes, R$ 4,043,768.85

                          AUDITORIA DE CONSOLIDAÇÃO                           

  Faturamento nas vendas (origem)  : R$    4,115,197.95
  Faturamento no consolidado       : R$    4,115,197.95
  Diferença                        : R$            0.00
  ✅ Valor conservado — nenhuma duplicação nos merges.
  ✅ Chave única: 1,473 clientes, um por linha.

  Cobertura por trilha:
    ├─ com vendas     : 1,379 clientes
    ├─ com financeiro : 1,346 clientes
    ├─ com comodato   : 1,280 clientes
    └─ core business  : 1,349 clientes  (chopp/chopeira)

  ℹ️  94 cliente(s) sem venda registrada têm PRIMEIRA_COMPRA,
      ULTIMA_COMPRA e DIAS_DESDE_* nulos — é a leitura correta, não
      um defe